In [1]:
#=====IMPORTS=====#
import csv, json, re, time
from collections import Counter
from datetime import date
from pathlib import Path

import requests

In [2]:
#####=====CONFIG=====####
HOUSE = "Commons"                 # "Commons" or "Lords"
START = date(2026, 10, 27)        # first sitting date to include
END   = date(2026, 10, 29)        # last sitting date to include
CONTACT = "your-email@example.com"

# --- RUN SIZE CONTROLS ---
MAX_DEBATES = 8                   # Set to a number for testing (e.g. 5), or None for the full dataset
TAKE  = 10                        # results per page (the API may cap this lower)
# -------------------------

DELAY = 1.0                       # seconds between requests, be polite
OUT_DIR = Path("hansard_output")
RAW_DIR = OUT_DIR / "raw_json"

# Titles to exclude (compared lowercase, stripped). Extend this as you find more.
BLOCKLIST = {
    "ways and means",
    "personal statement",
    "business of the house",
    "arrangement of business",
    "petitions",
    "business without debate",
    "point of order"
}
# --------------------------------------------------------

API = "https://hansard-api.parliament.uk"
HEADERS = {"User-Agent": f"research-script ({CONTACT})"}
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"{HOUSE}: {START} to {END}  ->  saving to {OUT_DIR.resolve()}")

Commons: 2026-10-27 to 2026-10-29  ->  saving to /content/hansard_output


In [3]:
####====HELPERS====####
session = requests.Session()
session.headers.update(HEADERS)

def get_json(url, params=None, retries=3):
    # GET a URL and return parsed JSON, with polite delay and simple retries.
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, params=params, timeout=30)
        except requests.RequestException as e:
            if attempt == retries:
                raise
            print(f"  network error ({e}); retry {attempt}/{retries - 1}")
            time.sleep(DELAY * attempt * 2)
            continue

        if r.status_code == 403:
            raise RuntimeError(f"403 Forbidden from {r.url}. Server={r.headers.get('Server')}, "
                               f"CF-Ray={r.headers.get('CF-Ray')}. Possibly bot protection.")
        if r.status_code in (429, 500, 502, 503, 504) and attempt < retries:
            print(f"  HTTP {r.status_code}; retry {attempt}/{retries - 1}")
            time.sleep(DELAY * attempt * 2)
            continue

        r.raise_for_status()
        time.sleep(DELAY)
        return r.json()

def strip_html(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()

In [4]:
####====FETCH METADATA====####
def fetch_debate_list():
    # Page through /search/debates for the configured house and dates.
    results, seen, skip = [], set(), 0

    for _ in range(500):                      # hard cap on pages, avoids infinite loops
        data = get_json(f"{API}/search/debates.json", {
            "queryParameters.house": HOUSE,
            "queryParameters.startDate": START.isoformat(),
            "queryParameters.endDate": END.isoformat(),
            "queryParameters.orderBy": "SittingDateAsc",
            "queryParameters.skip": skip,
            "queryParameters.take": TAKE,
        })
        batch = data.get("Results", [])
        total = data.get("TotalResultCount", 0)

        if skip == 0:
            print("TotalResultCount:", total)
            if total > 50_000:
                raise RuntimeError("Total is huge, so the date/house filters are probably not "
                                   "being applied. Check the parameter names in Swagger.")
        if not batch:
            break
        for r in batch:
            if r["DebateSectionExtId"] not in seen:
                seen.add(r["DebateSectionExtId"])
                results.append(r)
                if MAX_DEBATES is not None and len(results) >= MAX_DEBATES:
                    break

        skip += len(batch)
        limit_str = f"limit: {MAX_DEBATES}" if MAX_DEBATES is not None else "no limit"
        print(f"  fetched {len(results)} of {total} ({limit_str})")
        if skip >= total or (MAX_DEBATES is not None and len(results) >= MAX_DEBATES):
            break
    return results

all_results = fetch_debate_list()
json.dump(all_results, open(OUT_DIR / "list_raw.json", "w", encoding="utf-8"), indent=2)

dates = sorted({r["SittingDate"][:10] for r in all_results})
print(f"\n{len(all_results)} sections returned")
print("Dates covered:", dates)
print("Houses:", dict(Counter(r["House"] for r in all_results)))
print("DebateSection values:", dict(Counter(r["DebateSection"] for r in all_results)))

TotalResultCount: 106
  fetched 8 of 106 (limit: 8)

8 sections returned
Dates covered: ['2026-10-27']
Houses: {'Commons': 8}
DebateSection values: {'Commons Chamber': 8}


In [9]:
####====FILTERING====####
def keep(r):
    return (
        r["House"] == HOUSE
        and r["DebateSection"] == f"{HOUSE} Chamber"      # drops Westminster Hall etc.
        and START.isoformat() <= r["SittingDate"][:10] <= END.isoformat()
        and r["Title"].strip().lower() not in BLOCKLIST
    )

wanted = sorted((r for r in all_results if keep(r)),
                key=lambda r: (r["SittingDate"], r["Title"].strip()))
excluded = [r for r in all_results if not keep(r)]
debate_ids = [r["DebateSectionExtId"] for r in wanted]

print(f"KEEPING {len(wanted)}\n")
for r in wanted:
    print(f'  {r["SittingDate"][:10]}  {r["DebateSectionExtId"]}  {r["Title"].strip()}')

print(f"\nEXCLUDED {len(excluded)}\n")
for r in excluded:
    print(f'  {r["SittingDate"][:10]}  [{r["DebateSection"]}]  {r["Title"].strip()}')

KEEPING 5

  2026-10-27  565BC6CF-D5F8-4AFB-9D06-1CBD965141BF  Building Safety (Remediation of  Unsafe Cladding etc) Bill
  2026-10-27  6CFF3D07-DAC3-4D00-B63E-FA6180E4BBAA  Developer Contributions  (Infrastructure Delivery)
  2026-10-27  50764AC6-99E1-4169-A4ED-335EB488A6D0  Services for Children in Care
  2026-10-27  595A8127-E255-422E-AB60-30F8AD5D9C13  Wymondham Post Office
  2026-10-27  D51C150B-E116-40B8-B2AA-5160D86E1315  Yorkhill Fire Station

EXCLUDED 3

  2026-10-27  [Commons Chamber]  Petitions
  2026-10-27  [Commons Chamber]  Business without Debate
  2026-10-27  [Commons Chamber]  Point of Order


In [6]:
####====DOWNLOAD====####
summary, failures = [], []

for n, r in enumerate(wanted, 1):
    debate_id = r["DebateSectionExtId"]
    path = RAW_DIR / f'{r["SittingDate"][:10]}_{debate_id}.json'

    try:
        if path.exists():                                   # resume: skip already-downloaded
            data = json.load(open(path, encoding="utf-8"))
        else:
            data = get_json(f"{API}/debates/debate/{debate_id}.json")
            json.dump(data, open(path, "w", encoding="utf-8"), indent=2)
    except Exception as e:
        print(f"[{n}/{len(wanted)}] FAILED {debate_id} ({r['Title'].strip()}): {e}")
        failures.append(debate_id)
        continue

    items = data.get("Items", [])
    words = sum(len(strip_html(i.get("Value")).split()) for i in items)
    speakers = {i["MemberId"] for i in items if i.get("MemberId")}
    row = {
        "id": debate_id,
        "date": data["Overview"]["Date"][:10],
        "title": data["Overview"]["Title"].strip(),
        "n_items": len(items),
        "n_speakers": len(speakers),
        "n_children": len(data.get("ChildDebates", [])),
        "words": words,
        "file": path.name,
    }
    summary.append(row)
    print(f'[{n}/{len(wanted)}] {row["date"]} | {row["title"]} | '
          f'{row["n_items"]} items, {row["n_speakers"]} speakers, {row["words"]} words, '
          f'{row["n_children"]} children')

print(f"\nDone. {len(summary)} downloaded, {len(failures)} failed.")

[1/5] 2026-10-27 | Building Safety (Remediation of  Unsafe Cladding etc) Bill | 110 items, 36 speakers, 28895 words, 0 children
[2/5] 2026-10-27 | Developer Contributions  (Infrastructure Delivery) | 4 items, 1 speakers, 1392 words, 0 children
[3/5] 2026-10-27 | Services for Children in Care | 14 items, 4 speakers, 4931 words, 0 children
[4/5] 2026-10-27 | Wymondham Post Office | 2 items, 1 speakers, 201 words, 0 children
[5/5] 2026-10-27 | Yorkhill Fire Station | 1 items, 1 speakers, 258 words, 0 children

Done. 5 downloaded, 0 failed.


In [7]:
####====SAVE, EXPORT & INTEGRITY CHECK====####
if summary:
    csv_path = OUT_DIR / "debate_summary.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=summary[0].keys())
        w.writeheader()
        w.writerows(summary)
    print("Saved", csv_path)

if failures:
    print("Failed IDs (re-run the download cell to retry; finished ones are skipped):")
    for i in failures:
        print(" ", i)

# flags for debates that may be empty shells or procedural
for row in summary:
    if row["n_items"] == 0 and row["n_children"] > 0:
        print(f'Text may be in child sections: {row["title"]}')
    elif row["words"] < 300:
        print(f'Very short ({row["words"]} words): {row["title"]}')

Saved hansard_output/debate_summary.csv
Very short (201 words): Wymondham Post Office
Very short (258 words): Yorkhill Fire Station
